In [1]:
import math
from typing import Optional, Tuple, Union, Dict, List, Any

import torch
import torch.nn as nn

from transformers.cache_utils import Cache
from transformers.utils import add_start_docstrings, ModelOutput, logging
from transformers.modeling_utils import PreTrainedModel
from transformers import LlamaModel, AutoConfig
from transformers.configuration_utils import PretrainedConfig

from typing import TYPE_CHECKING
from dataclasses import dataclass

from llamafactory.model.loader import (
    register_autoclass, 
    init_adapter,
)

logger = logging.get_logger(__name__)

@dataclass
class CLSOutput(ModelOutput):
    loss: Optional[Union[torch.FloatTensor, Dict[str, torch.FloatTensor]]] = None
    hidden_states: Optional[Union[Tuple[torch.FloatTensor, ...], Dict[str, torch.FloatTensor]]] = None
    activations: Optional[Union[Tuple[torch.FloatTensor, ...], Dict[str, torch.FloatTensor]]] = None

if TYPE_CHECKING:
    from transformers import PreTrainedModel, PreTrainedTokenizer

    from ...hparams import FinetuningArguments, ModelArguments

CLS_START_DOCSTRING = r"""
    This model inherits from [`PreTrainedModel`]. Check the superclass documentation for the generic methods the
    library implements for all its model (such as downloading or saving, resizing the input embeddings, pruning heads
    etc.)

    This model is also a PyTorch [torch.nn.Module](https://pytorch.org/docs/stable/nn.html#torch.nn.Module) subclass.
    Use it as a regular PyTorch Module and refer to the PyTorch documentation for all matter related to general usage
    and behavior.

    Parameters:
        config ([`CLSConfig`]):
            Model configuration class with all the parameters of the model. Initializing with a config file does not
            load the weights associated with the model, only the configuration. Check out the
            [`~PreTrainedModel.from_pretrained`] method to load the model weights.
"""

class CLSConfig(PretrainedConfig):
    r"""
    This is the configuration class to store the configuration of a [`CLSModel`]. It is used to instantiate an CLS
    model according to the specified arguments, defining the model architecture. Instantiating a configuration with the
    defaults will yield a similar configuration to that of the CLS-7B.

    Configuration objects inherit from [`PretrainedConfig`] and can be used to control the model outputs. Read the
    documentation from [`PretrainedConfig`] for more information.


    Args:
        vocab_size (`int`, *optional*, defaults to 32000):
            Vocabulary size of the CLS model. Defines the number of different tokens that can be represented by the
            `inputs_ids` passed when calling [`CLSModel`]
        hidden_size (`int`, *optional*, defaults to 4096):
            Dimension of the hidden representations.
        num_hidden_layers (`int`, *optional*, defaults to 32):
            Number of hidden layers in the Transformer decoder.
        hidden_act (`str` or `function`, *optional*, defaults to `"silu"`):
            The non-linear activation function (function or string) in the decoder.
        initializer_range (`float`, *optional*, defaults to 0.02):
            The standard deviation of the truncated_normal_initializer for initializing all weight matrices.

    ```python
    >>> from transformers import CLSModel, CLSConfig

    >>> # Initializing a CLS CLS-7b style configuration
    >>> configuration = CLSConfig()

    >>> # Initializing a model from the CLS-7b style configuration
    >>> model = CLSModel(configuration)

    >>> # Accessing the model configuration
    >>> configuration = model.config
    ```"""

    model_type = "cls"

    def __init__(
        self,
        hidden_size:int=4096,
        num_labels:int=4,
        num_tasks:int=8,
        initializer_range:float=0.02,
        **kwargs,
    ):
        super().__init__(
            **kwargs,
        )
        self.num_labels= num_labels
        self.num_tasks = num_tasks
        self.hidden_size = hidden_size
        self.initializer_range = initializer_range

@add_start_docstrings(
    "The bare CLS Model outputting raw hidden-states without any specific head on top.",
    CLS_START_DOCSTRING,
)
class CLSPreTrainedModel(PreTrainedModel):
    config_class = CLSConfig
    base_model_prefix = "model"

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

class CLS(CLSPreTrainedModel):
    def __init__(
            self, 
            config: CLSConfig,
            **kwargs
            ):
        super().__init__(config)
        self.model_config = AutoConfig.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
        # self.model = load_model(tokenizer, model_args, finetuning_args, False, add_valuehead)
        self.model = LlamaModel.from_pretrained("/data/pretrained-models/meta/Llama-3.2-3B-Instruct")
        # patch_model(self.model, tokenizer, model_args, False, add_valuehead)
        self.model.requires_grad_(False)
        self.model.eval()
        print(self.config)
        self.task_score = nn.Linear(config.hidden_size, config.num_tasks, bias=False, device=self.model.device, dtype=self.model.dtype)
        self.label_score = nn.Linear(config.hidden_size, config.num_labels, bias=False, device=self.model.device, dtype=self.model.dtype)
        self.ignore_index = -100
        self.post_init()

    def forward(
        self,
        input_ids: torch.LongTensor = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[Union[Cache, List[torch.FloatTensor]]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        type_labels: Optional[torch.LongTensor] = None,
        task_labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
    ) -> Union[Dict, Tuple, torch.Tensor, CLSOutput]:
        with torch.no_grad():
            if input_ids is not None:
                batch_size = input_ids.shape[0]
            else:
                batch_size = inputs_embeds.shape[0]
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_values=past_key_values,
                inputs_embeds=inputs_embeds,
                use_cache=use_cache,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
                cache_position=cache_position,
            )
            hidden_states = outputs[0]

        if self.model_config.pad_token_id is None:
            sequence_lengths = -1
        else:
            if input_ids is not None:
                # if no pad token found, use modulo instead of reverse indexing for ONNX compatibility
                sequence_lengths = torch.eq(input_ids, self.model_config.pad_token_id).int().argmax(-1) - 1
                sequence_lengths = sequence_lengths % input_ids.shape[-1]
                sequence_lengths = sequence_lengths.to(hidden_states.device)
            else:
                sequence_lengths = -1

        eos_hidden_states = hidden_states[torch.arange(batch_size, device=hidden_states.device), sequence_lengths]
        label_scores = self.label_score(eos_hidden_states)
        task_scores = self.task_score(eos_hidden_states)
        if type_labels is not None and task_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(label_scores, type_labels) + loss_fct(task_scores, task_labels)
            return (loss, )
        else:
            label_logits = torch.softmax(label_scores, dim=-1)
            task_scores = torch.softmax(task_scores, dim=-1)
        return (label_logits, task_scores, )
        


/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC2/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


In [2]:
model = CLS.from_pretrained("/data/lihz/projects/instruct/IFT/ift/3b/cls_v2")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

CLSConfig {
  "_name_or_path": "/data/lihz/projects/instruct/IFT/ift/3b/cls_v2",
  "architectures": [
    "CLS"
  ],
  "hidden_size": 3072,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3"
  },
  "initializer_range": 0.02,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3
  },
  "model_type": "cls",
  "num_tasks": 10,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.43.2"
}



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
print(model.task_score.weight.shape)
print(model.label_score.weight.shape)

torch.Size([10, 3072])
torch.Size([4, 3072])


: 

In [23]:
from safetensors.torch import load_file

state_dict = load_file("/data/lihz/projects/instruct/IFT/ift/wcls_48_v2/models/model-00002-of-00002.safetensors")

In [30]:
state_dict = load_file("/data/lihz/projects/instruct/IFT/ift/wcls_48_v2/models/model-00001-of-00002.safetensors")

In [31]:
print(state_dict.keys())
# print(state_dict['label_score.weight'].shape)
# print(state_dict['task_score.weight'].shape)
size = 0
for k, v in state_dict.items():
    size += v.numel()

dict_keys(['model.model.embed_tokens.weight', 'model.model.layers.0.input_layernorm.weight', 'model.model.layers.0.mlp.down_proj.weight', 'model.model.layers.0.mlp.gate_proj.weight', 'model.model.layers.0.mlp.up_proj.weight', 'model.model.layers.0.post_attention_layernorm.weight', 'model.model.layers.0.self_attn.k_proj.weight', 'model.model.layers.0.self_attn.o_proj.weight', 'model.model.layers.0.self_attn.q_proj.weight', 'model.model.layers.0.self_attn.v_proj.weight', 'model.model.layers.1.input_layernorm.weight', 'model.model.layers.1.mlp.down_proj.weight', 'model.model.layers.1.mlp.gate_proj.weight', 'model.model.layers.1.mlp.up_proj.weight', 'model.model.layers.1.post_attention_layernorm.weight', 'model.model.layers.1.self_attn.k_proj.weight', 'model.model.layers.1.self_attn.o_proj.weight', 'model.model.layers.1.self_attn.q_proj.weight', 'model.model.layers.1.self_attn.v_proj.weight', 'model.model.layers.10.input_layernorm.weight', 'model.model.layers.10.mlp.down_proj.weight', 'mod

In [13]:
print(size)
# 7213584384
# 1123903488
# 2482888704

2482888704


In [15]:
(1123903488 + 2482888704) * 2 == 7213584384

True

In [4]:
import json

In [18]:
with open("/data/lihz/projects/instruct/IFT/ift/wcls_48_v2/models/model.safetensors.index.json", 'r') as f:
    json_object = json.load(f)

In [21]:
state_dict = load_file("/data/pretrained-models/meta/Llama-3.2-3B-Instruct/model-00001-of-00002.safetensors")
print(state_dict.keys())

dict_keys(['model.embed_tokens.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.1.post_attention_layernorm.weight', 'model.layers.1.self_attn.k_proj.weight', 'model.layers.1.self_attn.o_proj.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.1.self_attn.v_proj.weight', 'model.layers.10.input_layernorm.weight', 'model.layers.10.mlp.down_proj.weight', 'model.layers.10.mlp.gate_proj.weight', 'model.layers.10.mlp.up_proj.weight', 'model.layers.10.post_attention_layernorm.weight', '

In [45]:
print(json_object.keys())

dict_keys(['metadata', 'weight_map'])


In [19]:
print(json_object['metadata'])
print(json_object['weight_map'])

{'total_size': 6425499648}
{'model.embed_tokens.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.input_layernorm.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.mlp.down_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.mlp.gate_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.mlp.up_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.post_attention_layernorm.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.self_attn.k_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.self_attn.o_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.self_attn.q_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.self_attn.v_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.1.input_layernorm.weight': 'model-00001-of-00002.safetensors', 'model.layers.1.mlp.down_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.1.mlp.gate_proj.weight': 'model-00001-of-00

In [51]:
print(json_object['metadata'])
print(json_object['weight_map'])

{'total_size': 7213584384}
{'lm_head.weight': 'model-00002-of-00002.safetensors', 'model.embed_tokens.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.input_layernorm.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.mlp.down_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.mlp.gate_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.mlp.up_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.post_attention_layernorm.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.self_attn.k_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.self_attn.o_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.self_attn.q_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.0.self_attn.v_proj.weight': 'model-00001-of-00002.safetensors', 'model.layers.1.input_layernorm.weight': 'model-00001-of-00002.safetensors', 'model.layers.1.mlp.down_proj.weight': 'model-00001-of-00002.safetensors', 'mo

In [50]:
json_object['weight_map'] = {
    k[6:]:v
    for k, v in json_object['weight_map'].items()
    if 'score' not in k
}

In [16]:
# import json
# with open("/data/lihz/projects/instruct/IFT/ift/wcls_48_v2/models/model.safetensors.index.json", 'w') as f:
#     json.dump(json_object, f)

NameError: name 'json_object' is not defined

In [8]:
print(json_object['weight_map'])

{'label_score.weight': 'model-00002-of-00002.safetensors', 'model.lm_head.weight': 'model-00002-of-00002.safetensors', 'model.model.embed_tokens.weight': 'model-00001-of-00002.safetensors', 'model.model.layers.0.input_layernorm.weight': 'model-00001-of-00002.safetensors', 'model.model.layers.0.mlp.down_proj.weight': 'model-00001-of-00002.safetensors', 'model.model.layers.0.mlp.gate_proj.weight': 'model-00001-of-00002.safetensors', 'model.model.layers.0.mlp.up_proj.weight': 'model-00001-of-00002.safetensors', 'model.model.layers.0.post_attention_layernorm.weight': 'model-00001-of-00002.safetensors', 'model.model.layers.0.self_attn.k_proj.weight': 'model-00001-of-00002.safetensors', 'model.model.layers.0.self_attn.o_proj.weight': 'model-00001-of-00002.safetensors', 'model.model.layers.0.self_attn.q_proj.weight': 'model-00001-of-00002.safetensors', 'model.model.layers.0.self_attn.v_proj.weight': 'model-00001-of-00002.safetensors', 'model.model.layers.1.input_layernorm.weight': 'model-0000

In [40]:
# state_dict = load_file("/data/lihz/projects/instruct/IFT/ift/wcls_48_v2/models/model-00002-of-00002.safetensors")

In [41]:
state_dict = {
    k[6:]: v
    for k, v in state_dict.items()
    if "score" not in k
}

In [1]:
from safetensors.torch import save_file

In [2]:
# save_file(state_dict, "/data/lihz/projects/instruct/IFT/ift/wcls_48_v2/models/model-00002-of-00002.safetensors")

In [16]:
from safetensors.torch import save_file
from safetensors import safe_open

# with safe_open("/data/lihz/projects/instruct/IFT/ift/rwcls_48_v6/model-00002-of-00002.safetensors", framework="pt") as f:
# with safe_open("/data/lihz/projects/instruct/IFT/ift/rwcls_48_v6_008/model-00002-of-00002.safetensors", framework="pt") as f:
with safe_open(
    # "/data/lihz/projects/instruct/IFT/ift/rwcls_48_v6_008_1B/model.safetensors",
    # "/data/lihz/projects/instruct/IFT/ift/rwcls_48_v6_008_1B/model.safetensors", 
    # "/data/lihz/projects/instruct/IFT/ift/math/1b-e2-inst/model.safetensors",
    "/data/lihz/projects/instruct/IFT/ift/math/8b-e2-base/model-00002-of-00004.safetensors",
    # "/data/lihz/projects/instruct/IFT/ift/code/3b-e2-inst/model-00002-of-00002.safetensors",
    framework="pt") as f:
# with safe_open("/data/lihz/projects/instruct/IFT/ift/math/3b-e2/model-00001-of-00002.safetensors", 
#                framework="pt") as f:
    meta_data = f.metadata()
    state_dict = { k[6:]:f.get_tensor(k) for k in f.keys() if 'score' not in k}

In [17]:
print(meta_data)

{'format': 'pt'}


In [18]:
# save_file(state_dict, "/data/lihz/projects/instruct/IFT/ift/rwcls_48_v6/models/model-00002-of-00002.safetensors", metadata=meta_data)
# save_file(state_dict, "/data/lihz/projects/instruct/IFT/ift/rwcls_48_v6_008/models/model-00002-of-00002.safetensors", metadata=meta_data)
# save_file(state_dict, "/data/lihz/projects/instruct/IFT/ift/rwcls_48_v6_008_1B/models/model.safetensors", metadata=meta_data)
save_file(state_dict, 
          # "/data/lihz/projects/instruct/IFT/ift/math/1b-e2-inst/models/model.safetensors", 
          "/data/lihz/projects/instruct/IFT/ift/math/8b-e2-base/models/model-00002-of-00004.safetensors", 
          # "/data/lihz/projects/instruct/IFT/ift/code/3b-e2-inst/models/model-00002-of-00002.safetensors", 
          metadata=meta_data)

In [1]:
import torch
import torch_npu
import torch.nn as nn

# 假设 target 是一个包含类别标签的 LongTensor
target = torch.tensor([0, 0, 1, 2, 0, 1])  # 类别 0, 0, 1, 2, 0, 1

# 设置权重
weights = torch.tensor([1.0, 10.0, 100.0])  # 类别 A, B, C 的权重

# 创建 CrossEntropyLoss 实例，并传入 weight 参数
loss_func = nn.CrossEntropyLoss(weight=weights)

# 假设 output 是模型的输出，形状为 (batch_size, num_classes)
output = torch.randn(6, 3) # 随机生成output模拟模型输出

loss = loss_func(output, target)

print(loss)

tensor(1.1768)


/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC2/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


In [ ]:
import torch.nn.functional as F

class MyCrossEntropyLoss(nn.Module):
    def __init__(self, weight=None, reduction='mean'):
        super(MyCrossEntropyLoss, self).__init__()
        self.weight:torch.Tensor = weight
        self.reduction = reduction

    def forward(self, input, target):
        batch_size = input.size(0)
        num_classes = input.size(1)

        log_probs = F.log_softmax(input, dim=1)
        target = target.long()
        log_probs_target = log_probs.gather(1, target.unsqueeze(1)).squeeze(1)

        if self.weight is not None:
            weights = self.weight.gather(0, target)
            weighted_loss = -log_probs_target * weights

            if self.reduction == 'mean':
                loss = weighted_loss.sum() / weights.sum()  # 正确的加权平均
            elif self.reduction == 'sum':
                loss = weighted_loss.sum()
            elif self.reduction == 'none':
                loss = weighted_loss
            else:
                raise ValueError("Invalid reduction option: {}".format(self.reduction))
        else:
            loss = -log_probs_target.mean() if self.reduction == 'mean' else -log_probs_target.sum() if self.reduction == 'sum' else -log_probs_target

        return loss

# 测试代码
batch_size = 4
num_classes = 3

# 随机生成输入和目标
input = torch.randn(batch_size, num_classes)
target = torch.randint(0, num_classes, (batch_size,))

# 设置权重
weights = torch.tensor([1.0, 10.0, 100.0])

# 使用 PyTorch 的 CrossEntropyLoss
torch_loss_func = nn.CrossEntropyLoss(weight=weights, reduction='mean')
torch_loss = torch_loss_func(input, target)

# 使用自定义的 CrossEntropyLoss
my_loss_func = MyCrossEntropyLoss(weight=weights, reduction='mean')
my_loss = my_loss_func(input, target)

print("PyTorch CrossEntropyLoss:", torch_loss)
print("My CrossEntropyLoss:", my_loss)

# 不使用权重进行比较
torch_loss_func_no_weight = nn.CrossEntropyLoss(reduction='mean')
torch_loss_no_weight = torch_loss_func_no_weight(input, target)

my_loss_func_no_weight = MyCrossEntropyLoss(reduction='mean')
my_loss_no_weight = my_loss_func_no_weight(input, target)

print("PyTorch CrossEntropyLoss (no weight):", torch_loss_no_weight)
print("My CrossEntropyLoss (no weight):", my_loss_no_weight)

# 测试reduction='none'的情况
torch_loss_func_none = nn.CrossEntropyLoss(weight=weights, reduction='none')
torch_loss_none = torch_loss_func_none(input, target)

my_loss_func_none = MyCrossEntropyLoss(weight=weights, reduction='none')
my_loss_none = my_loss_func_none(input, target)

print("PyTorch CrossEntropyLoss (reduction=none):", torch_loss_none)
print("My CrossEntropyLoss (reduction=none):", my_loss_none)

print("Are the losses close?", torch.allclose(torch_loss, my_loss))
print("Are the losses close (no weight)?", torch.allclose(torch_loss_no_weight, my_loss_no_weight))
print("Are the losses close (reduction=none)?", torch.allclose(torch_loss_none, my_loss_none))


PyTorch CrossEntropyLoss: tensor(1.4103)
My CrossEntropyLoss: tensor(1.4103)
PyTorch CrossEntropyLoss (no weight): tensor(0.8878)
My CrossEntropyLoss (no weight): tensor(0.8878)
PyTorch CrossEntropyLoss (reduction=none): tensor([  0.7024,   7.2007,   5.5709, 157.1700])
My CrossEntropyLoss (reduction=none): tensor([  0.7024,   7.2007,   5.5709, 157.1700])
Are the losses close? True
Are the losses close (no weight)? True
Are the losses close (reduction=none)? True


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MyCrossEntropyLoss(nn.Module):
    def __init__(self, weight=None, ignore_index=-100, reduction='mean'):
        super(MyCrossEntropyLoss, self).__init__()
        self.weight = weight
        self.ignore_index = ignore_index
        self.reduction = reduction

    def forward(self, input, target):
        batch_size = input.size(0)
        num_classes = input.size(1)

        log_probs = F.log_softmax(input, dim=1)
        target = target.long()

        # 创建一个和target相同shape的loss tensor，初始化为0
        loss = torch.zeros_like(target, dtype=input.dtype, device=input.device)

        # 创建一个mask，标记需要计算loss的样本
        mask = target != self.ignore_index

        # 使用mask过滤目标和对数概率
        masked_target = target[mask]
        masked_log_probs = log_probs[mask]

        if masked_target.numel() > 0: # 只有当有有效target的时候才计算loss
            log_probs_target = masked_log_probs.gather(1, masked_target.unsqueeze(1)).squeeze(1)

            if self.weight is not None:
                weights = self.weight.gather(0, masked_target)
                weighted_loss = -log_probs_target * weights
            else:
                weighted_loss = -log_probs_target
            
            # 将计算得到的loss放到预先创建的loss tensor中正确的位置上
            loss[mask] = weighted_loss

        if self.reduction == 'mean':
            loss = loss.sum()/(mask.sum() if mask.sum() > 0 else 1) # 这里需要判断mask.sum()是否为0，避免除0错误
        elif self.reduction == 'sum':
            loss = loss.sum()
        elif self.reduction == 'none':
            pass # loss已经是每个元素的loss了，不需要再处理
        else:
            raise ValueError("Invalid reduction option: {}".format(self.reduction))

        return loss

# 测试代码
batch_size = 4
num_classes = 3
ignore_index = -100 # 设置忽略索引

input = torch.randn(batch_size, num_classes)
target = torch.tensor([0, ignore_index, 2, 1]) # target中包含ignore_index

# 使用 PyTorch 的 CrossEntropyLoss
torch_loss_func = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.0, 3.0]), ignore_index=ignore_index, reduction='mean')
torch_loss = torch_loss_func(input, target)

# 使用自定义的 CrossEntropyLoss
my_loss_func = MyCrossEntropyLoss(weight=torch.tensor([1.0, 2.0, 3.0]), ignore_index=ignore_index, reduction='mean')
my_loss = my_loss_func(input, target)

print("PyTorch CrossEntropyLoss:", torch_loss)
print("My CrossEntropyLoss:", my_loss)
print("Are the losses close?", torch.allclose(torch_loss, my_loss))


# 测试所有target都被ignore的情况
target_all_ignore = torch.tensor([ignore_index, ignore_index, ignore_index, ignore_index])
torch_loss_all_ignore = nn.CrossEntropyLoss(ignore_index=ignore_index, reduction='mean')(input, target_all_ignore)
my_loss_all_ignore = MyCrossEntropyLoss(ignore_index=ignore_index, reduction='mean')(input, target_all_ignore)
print("PyTorch CrossEntropyLoss (all ignore):", torch_loss_all_ignore)
print("My CrossEntropyLoss (all ignore):", my_loss_all_ignore)
print("Are the losses close (all ignore)?", torch.allclose(torch_loss_all_ignore, my_loss_all_ignore))

PyTorch CrossEntropyLoss: tensor(0.9652)
My CrossEntropyLoss: tensor(1.9303)
Are the losses close? False
PyTorch CrossEntropyLoss (all ignore): tensor(nan)
My CrossEntropyLoss (all ignore): tensor(0.)
Are the losses close (all ignore)? False


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ... (MyCrossEntropyLoss 的定义，与上次的最终版本相同)

# 测试代码
def test_cross_entropy_loss():
    batch_size = 4
    num_classes = 3
    ignore_index = -100
    weights = torch.tensor([1.0, 2.0, 3.0])

    test_cases = [
        {"reduction": "mean", "weight": None, "target": torch.tensor([0, 1, 2, 0])},
        {"reduction": "sum", "weight": None, "target": torch.tensor([0, 1, 2, 0])},
        {"reduction": "none", "weight": None, "target": torch.tensor([0, 1, 2, 0])},
        {"reduction": "mean", "weight": weights, "target": torch.tensor([0, 1, 2, 0])},
        {"reduction": "sum", "weight": weights, "target": torch.tensor([0, 1, 2, 0])},
        {"reduction": "none", "weight": weights, "target": torch.tensor([0, 1, 2, 0])},
        {"reduction": "mean", "weight": None, "ignore_index": ignore_index, "target": torch.tensor([0, ignore_index, 2, 1])},
        {"reduction": "sum", "weight": None, "ignore_index": ignore_index, "target": torch.tensor([0, ignore_index, 2, 1])},
        {"reduction": "none", "weight": None, "ignore_index": ignore_index, "target": torch.tensor([0, ignore_index, 2, 1])},
        {"reduction": "mean", "weight": weights, "ignore_index": ignore_index, "target": torch.tensor([0, ignore_index, 2, 1])},
        {"reduction": "sum", "weight": weights, "ignore_index": ignore_index, "target": torch.tensor([0, ignore_index, 2, 1])},
        {"reduction": "none", "weight": weights, "ignore_index": ignore_index, "target": torch.tensor([0, ignore_index, 2, 1])},
        {"reduction": "mean", "weight": weights, "ignore_index": ignore_index, "target": torch.tensor([ignore_index, ignore_index, ignore_index, ignore_index])},
        {"reduction": "sum", "weight": weights, "ignore_index": ignore_index, "target": torch.tensor([ignore_index, ignore_index, ignore_index, ignore_index])},
        {"reduction": "none", "weight": weights, "ignore_index": ignore_index, "target": torch.tensor([ignore_index, ignore_index, ignore_index, ignore_index])},
    ]

    for case in test_cases:
        input = torch.randn(batch_size, num_classes, requires_grad=True)
        target = case["target"]
        reduction = case["reduction"]
        weight = case.get("weight")  # 使用 .get() 方法，如果键不存在则返回 None
        ignore_index = case.get("ignore_index")

        if ignore_index is not None:
            torch_loss_func = nn.CrossEntropyLoss(weight=weight, ignore_index=ignore_index, reduction=reduction)
            my_loss_func = MyCrossEntropyLoss(weight=weight, ignore_index=ignore_index, reduction=reduction)
        else:
            torch_loss_func = nn.CrossEntropyLoss(weight=weight, reduction=reduction)
            my_loss_func = MyCrossEntropyLoss(weight=weight, reduction=reduction)

        torch_loss = torch_loss_func(input, target)
        my_loss = my_loss_func(input, target)

        print(f"Reduction: {reduction}, Weight: {weight is not None}, Ignore Index: {ignore_index is not None}")
        print("PyTorch CrossEntropyLoss:", torch_loss)
        print("My CrossEntropyLoss:", my_loss)
        print("Are the losses close?", torch.allclose(torch_loss, my_loss, equal_nan=True))

        torch_loss.backward()
        my_loss.backward()

        print("input gradients close?", torch.allclose(input.grad, input.grad, equal_nan=True))

        input.grad.zero_()
        input.grad.zero_()
        print("-" * 20)

test_cross_entropy_loss()

Reduction: mean, Weight: False, Ignore Index: False
PyTorch CrossEntropyLoss: tensor(1.0773, grad_fn=<NllLossBackward0>)
My CrossEntropyLoss: tensor(1.0773, grad_fn=<DivBackward0>)
Are the losses close? True
input gradients close? True
--------------------
Reduction: sum, Weight: False, Ignore Index: False
PyTorch CrossEntropyLoss: tensor(6.7775, grad_fn=<NllLossBackward0>)
My CrossEntropyLoss: tensor(6.7775, grad_fn=<SumBackward0>)
Are the losses close? True
input gradients close? True
--------------------
Reduction: none, Weight: False, Ignore Index: False
PyTorch CrossEntropyLoss: tensor([1.7306, 2.8654, 1.6490, 0.3340], grad_fn=<NllLossBackward0>)
My CrossEntropyLoss: tensor([1.7306, 2.8654, 1.6490, 0.3340], grad_fn=<IndexPutBackward0>)
Are the losses close? True


RuntimeError: grad can be implicitly created only for scalar outputs